## Wales Income Tax

#### Data and calculations

#### Current data is for 2023-24 tax year: [Welsh Income Tax Outturn Statistical Tables — 2023 to 2024](https://assets.publishing.service.gov.uk/media/685bd4d6c07c71e5a87097e2/Welsh_Income_Tax_Outturn_2023-2024.ods)

### Definitions:

**Outturn**: This is the term used to give how much income tax was collected in a given tax year

**Non-Saving Non-Dividen Income - NSND** - this is the money income tax is collected on, and includes earnings from employment, pensions, profits from self-employed sources and property

In **Wales**, The Government of Wales Act 2006 enables the Senedd (Welsh Parliament) to set a Welsh basic, higher, and additional rate of Income Tax to apply to NSND income. It was first introduced in the 2019-20 tax year.

It works like this: Whatever the three tax rates set by the UK government, that rate is **reduced** by 10 pence in the pound, as it is applied in Wales. So, a basic income tax rate of 20% in England becomes 10% in Wales.
The Welsh government and Senedd can then increase or decrease its element. Since the introduction of the powers in 2019, the Welsh government has mirrored the rates set by the UK government - so it's additional element has remained at +10% for basic rate, mirroring the 20% for England. **NOTE** Scotland has an entirely different band structure for income tax.

### Import the data

In [2]:
#use requests to get the dataset from url
import requests

In [3]:
#set url variable as html address for file
url = "https://assets.publishing.service.gov.uk/media/685bd4d6c07c71e5a87097e2/Welsh_Income_Tax_Outturn_2023-2024.ods"
filename = "Welsh_Income_Tax_Outturn_2023-2024.ods"

#use requests to get dataset - in variable response
response = requests.get(url)


#"wb" means: w = write mode (create or overwrite a file) b = binary mode (write bytes, not text)
with open(filename, "wb") as f:
    f.write(response.content)

print("Downloaded:", filename)


Downloaded: Welsh_Income_Tax_Outturn_2023-2024.ods


### Import the downloaded dataset into a dataframe

In [2]:
import pandas as pd
import numpy as np

In [3]:
#reads the .ods file as an excel file, and reads the individual sheets

xls = pd.ExcelFile("Welsh_Income_Tax_Outturn_2023-2024.ods", engine="odf")
print(xls.sheet_names)


sheets = {}

for name in xls.sheet_names:
    if name == "Table_3":
        # Skip rows 0–8, promote row 9 to header
        df = pd.read_excel(
            xls,
            sheet_name=name,
            engine="odf",
            skiprows=list(range(0, 9)),  # skip rows 0 to 7
            header=0                     # row 9 becomes the header
        )
    else:
        # Default behaviour for all other sheets
        df = pd.read_excel(xls, sheet_name=name, engine="odf")

    sheets[name] = df



['Cover_Sheet', 'Contents', 'Table_1', 'Table_2', 'Table_3', 'Table_4', 'Table_5', 'Notes']


In [5]:

#drops extraneous metadata from top of table sheet 3
sheets["Table_3"] = sheets["Table_3"].replace("[Not Applicable]", np.nan)
#drops last extraneous row from table 3
sheets["Table_3"] = sheets["Table_3"].drop(sheets["Table_3"].index[-1])


In [6]:
sheets["Table_3"]

,Taxpayer,Tax year,Tax band,Number of taxpayers,Basic rate taxpayers,Higher rate taxpayers,Additional rate taxpayers,All taxpayers
0,Welsh,2019-20,Basic Rate,1244900.0,1393.0,342.0,15.0,1750.0
1,Welsh,2019-20,Higher Rate,92000.0,NaN,185.0,46.0,231.0
2,Welsh,2019-20,Additional Rate,4200.0,NaN,NaN,45.0,45.0
3,Welsh,2019-20,All bands,1341100.0,1393.0,527.0,107.0,2026.0
4,Welsh,2020-21,Basic Rate,1260700.0,1443.0,369.0,16.0,1828.0
5,Welsh,2020-21,Higher Rate,99800.0,NaN,201.0,47.0,248.0
6,Welsh,2020-21,Additional Rate,4300.0,NaN,NaN,46.0,46.0
7,Welsh,2020-21,All bands,1364700.0,1443.0,570.0,109.0,2123.0
8,Welsh,2021-22,Basic Rate,1309000.0,1564.0,429.0,20.0,2013.0
9,Welsh,2021-22,Higher Rate,114700.0,NaN,230.0,59.0,289.0


In [7]:
df_welsh_tax = sheets["Table_3"]

In [10]:

with pd.ExcelWriter("Welsh_taxation.xlsx", engine="xlsxwriter") as writer:
    df_welsh_tax.to_excel(writer, sheet_name="Sheet1", index=False)
